In [51]:
# import sys
# !{sys.executable} -m pip install -q -U \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-google-genai \
#     faiss-cpu>=1.9.0 \
#     sentence-transformers \
#     ragas \
#     datasets \
#     langchain-text-splitters \
#     google-cloud-aiplatform \
#     langchain-google-vertexai

In [52]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Setting up API Key and Models
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Increase the timeout for the LLM to prevent TimeoutErrors during RAGAS evaluation
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, timeout=900)
embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
judge_llm = llm # LLM assigned for RAGAS evaluation

In [53]:
from langchain_core.callbacks.base import BaseCallbackHandler
from typing import Any, Dict, List, Union

class TokenCountingCallback(BaseCallbackHandler):
    def __init__(self):
        self.llm_input_tokens = 0
        self.llm_output_tokens = 0
        self.embedding_tokens = 0
        self.llm_calls = 0
        self.embedding_calls = 0
        print("TokenCountingCallback initialized.")

    def on_llm_start(self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any) -> None:
        self.llm_calls += 1
        # Approximate input tokens by character count (can be more precise with tokenizer)
        input_tokens_current_call = sum(len(p) for p in prompts)
        self.llm_input_tokens += input_tokens_current_call
        print(f"on_llm_start: LLM Call #{self.llm_calls}, Input Tokens: {input_tokens_current_call}")

    def on_llm_end(self, response: Any, **kwargs: Any) -> None:
        # `response.llm_output.token_usage` might be available for some models
        # For Gemini, often response.usage_metadata is not directly here.
        # We'll approximate output tokens by character count, or check the response object
        output_tokens_current_call = sum(len(g.text) for g in response.generations[0] if g.text is not None)
        self.llm_output_tokens += output_tokens_current_call
        print(f"on_llm_end: LLM Output Tokens: {output_tokens_current_call}")

    def on_embedding_start(self, serialized: Dict[str, Any], texts: List[str], **kwargs: Any) -> Any:
        self.embedding_calls += 1
        # Approximate embedding tokens by character count
        tokens_this_call = sum(len(t) for t in texts)
        self.embedding_tokens += tokens_this_call
        print(f"on_embedding_start: Embedding Call #{self.embedding_calls}, Texts: {len(texts)}, Tokens: {tokens_this_call}, Total Embedding Tokens: {self.embedding_tokens}")

    def on_embedding_end(self, response: List[List[float]], **kwargs: Any) -> Any:
        # GoogleGenerativeAIEmbeddings don't directly return token usage in on_embedding_end
        # We rely on on_embedding_start approximation for now or inspect client logs
        print(f"on_embedding_end: Embedding call finished.")
        pass

    def get_token_counts(self):
        return {
            "llm_input_tokens": self.llm_input_tokens,
            "llm_output_tokens": self.llm_output_tokens,
            "llm_total_tokens": self.llm_input_tokens + self.llm_output_tokens,
            "llm_calls": self.llm_calls,
            "embedding_tokens": self.embedding_tokens,
            "embedding_calls": self.embedding_calls,
        }

# Initialize the callback handler
token_counter = TokenCountingCallback()

# Re-initialize llm and embedder with the callback
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, timeout=900, callbacks=[token_counter])
embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", callbacks=[token_counter])
judge_llm = llm # LLM assigned for RAGAS evaluation

print("TokenCountingCallback initialized and attached to LLM and Embedder.")

TokenCountingCallback initialized.
TokenCountingCallback initialized and attached to LLM and Embedder.


After running the relevant cells (e.g., knowledge base creation, query generation, RAGAS evaluation) that use the `llm` and `embedder` objects, you can then retrieve the token counts:


In [54]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Audit data for the knowledge base
audit_data = [
    Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
    Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
    Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
    Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
    Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
    Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
    Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
    Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8}),
    Document(page_content="The new firmware v3.0 has been successfully tested in the North American cluster.", metadata={"id": 9}),
    Document(page_content="Budget allocation for Q1 also includes a 10% reserve for unexpected cloud egress costs.", metadata={"id": 10}),
    Document(page_content="Anisha suggested moving OCR processing to an asynchronous queue using RabbitMQ.", metadata={"id": 11}),
    Document(page_content="Legacy Scan-X machines have a known overheating issue when processing over 100 pages.", metadata={"id": 12}),
    Document(page_content="Compliance team noted that OCR errors are leading to incorrect data in customer KYC files.", metadata={"id": 13}),
    Document(page_content="The upgrade project is codenamed 'Project Vision' and is led by the MLOps Core team.", metadata={"id": 14}),
    Document(page_content="Handwritten PDF recognition accuracy dropped to 62% in the last batch test.", metadata={"id": 15}),
    Document(page_content="Security audit found that legacy firmware v2.1 has three unpatched vulnerabilities.", metadata={"id": 16}),
    Document(page_content="Training data for the new OCR model includes 50,000 samples of handwritten European scripts.", metadata={"id": 17}),
    Document(page_content="The hardware vendor 'OptiScan' has been notified about the hardware failures.", metadata={"id": 18}),
    Document(page_content="A temporary patch was deployed on Monday to reduce memory leaks during batch processing.", metadata={"id": 19}),
    Document(page_content="Q2 Roadmap: Complete migration of all legacy branches to the centralized MLOps platform.", metadata={"id": 20})
]

# Extract texts and metadatas from the documents
texts = [doc.page_content for doc in audit_data]
metadatas = [doc.metadata for doc in audit_data]

# Manually call on_embedding_start to track embedding calls and tokens
token_counter.on_embedding_start(serialized={}, texts=texts)
embeddings = embedder.embed_documents(texts)

# Create VectorStore from the pre-computed embeddings and original texts/metadatas
# The 'embedding' parameter is still required for future query embeddings.
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings)),
    embedding=embedder,
    metadatas=metadatas
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})
print("Vector Store status: READY")

on_embedding_start: Embedding Call #1, Texts: 20, Tokens: 1670, Total Embedding Tokens: 1670
Vector Store status: READY


In [55]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Stage 2: Cross-Encoder Initialization
cross_encoder_model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=3)

# Defining Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

# RAG Chain Setup
template = """Answer the question based ONLY on the following context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

advanced_rerank_chain = (
    {"context": compression_retriever, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)
print("Rerank Chain status: READY")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rerank Chain status: READY


In [56]:
# Test Dataset with Ground Truths
golden_dataset = [
    {
        "question": "What is the OCR failure rate in European legacy branches?",
        "ground_truth": "The OCR failure rate is 15%."
    },
    {
        "question": "Why do OCR failures peak on Tuesdays?",
        "ground_truth": "Due to weekly bulk-batch processing of handwritten PDFs."
    },
    {
        "question": "What is the codename for the upgrade project?",
        "ground_truth": "The project codename is 'Project Vision'."
    }
]

# Dictionary to store results for RAGAS
evaluation_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("Running pipeline on Golden Dataset...")

for item in golden_dataset:
    query = item["question"]

    # Retrieve relevant documents
    retrieved_docs = compression_retriever.invoke(query)
    context_list = [doc.page_content for doc in retrieved_docs]

    # Generate response
    response = advanced_rerank_chain.invoke(query)

    # Append to evaluation set
    evaluation_data["question"].append(query)
    evaluation_data["contexts"].append(context_list)
    evaluation_data["answer"].append(response)
    evaluation_data["ground_truth"].append(item["ground_truth"])

print(f"Data generation complete: {len(evaluation_data['question'])} samples.")

Running pipeline on Golden Dataset...
on_llm_start: LLM Call #1, Input Tokens: 651
on_llm_end: LLM Output Tokens: 56
on_llm_start: LLM Call #2, Input Tokens: 648
on_llm_end: LLM Output Tokens: 86
on_llm_start: LLM Call #3, Input Tokens: 647
on_llm_end: LLM Output Tokens: 14
Data generation complete: 3 samples.


The `ModuleNotFoundError` for `langchain_community.chat_models.vertexai` indicates a compatibility issue with how `ragas` tries to import VertexAI components, even when using a different LLM. To resolve this, we'll create dummy modules and classes for `vertexai` in `sys.modules` to satisfy `ragas`'s internal import checks.

In [57]:
import sys

# Create dummy classes that ragas might try to instantiate
class DummyChatVertexAI:
    def __init__(self, *args, **kwargs):
        pass

class DummyVertexAI:
    def __init__(self, *args, **kwargs):
        pass

# Create a dummy module object dynamically
dummy_vertexai_module = type(sys)("dummy_vertexai_module")
dummy_vertexai_module.ChatVertexAI = DummyChatVertexAI
dummy_vertexai_module.VertexAI = DummyVertexAI

# Insert the dummy module into sys.modules at the paths ragas expects
sys.modules['langchain_community.chat_models.vertexai'] = dummy_vertexai_module
sys.modules['langchain_community.llms.vertexai'] = dummy_vertexai_module

print("Dummy 'vertexai' modules created to bypass ragas import issue.")

Dummy 'vertexai' modules created to bypass ragas import issue.


In [58]:
from datasets import Dataset
from ragas import evaluate
from ragas.llms.base import LangchainLLMWrapper

# importing new objects
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision
)

# 1. Intialize the metrics here
faithfulness_obj = Faithfulness()
answer_relevancy_obj = AnswerRelevancy()
context_precision_obj = ContextPrecision()

if len(evaluation_data["question"]) > 0:
    ragas_dataset = Dataset.from_dict(evaluation_data)

    print("\n[System] Calculating RAGAS Metrics with Initialized Objects...")

    results = evaluate(
        dataset=ragas_dataset,
        metrics=[
            faithfulness_obj,
            answer_relevancy_obj,
            context_precision_obj
        ],
        llm=LangchainLLMWrapper(judge_llm),
        embeddings=embedder
    )

    import pandas as pd
    results_df = results.to_pandas()
    display(results_df)
else:
    print("❌ Error: evaluation_data is empty!")


[System] Calculating RAGAS Metrics with Initialized Objects...


/tmp/ipykernel_2188/3094779178.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_2188/3094779178.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipykernel_2188/3094779178.py:6: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/tmp/ipykernel_2188/3094779178.py:29: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

on_llm_start: LLM Call #4, Input Tokens: 1766
on_llm_start: LLM Call #5, Input Tokens: 1397
on_llm_start: LLM Call #6, Input Tokens: 4125
on_llm_start: LLM Call #7, Input Tokens: 1776
on_llm_start: LLM Call #8, Input Tokens: 1427
on_llm_start: LLM Call #9, Input Tokens: 4139
on_llm_start: LLM Call #10, Input Tokens: 1712
on_llm_start: LLM Call #11, Input Tokens: 1355
on_llm_start: LLM Call #12, Input Tokens: 4130
on_llm_end: LLM Output Tokens: 89


on_llm_end: LLM Output Tokens: 221
on_llm_end: LLM Output Tokens: 90
on_llm_end: LLM Output Tokens: 99


on_llm_start: LLM Call #13, Input Tokens: 4143
on_llm_start: LLM Call #14, Input Tokens: 3856
on_llm_start: LLM Call #15, Input Tokens: 1776
on_llm_start: LLM Call #16, Input Tokens: 1397
on_llm_start: LLM Call #17, Input Tokens: 4130
on_llm_start: LLM Call #18, Input Tokens: 1766
on_llm_start: LLM Call #19, Input Tokens: 4125
on_llm_end: LLM Output Tokens: 570
on_llm_start: LLM Call #20, Input Tokens: 3856
on_llm_start: LLM Call #21, Input Tokens: 4143
on_llm_end: LLM Output Tokens: 497
on_llm_start: LLM Call #22, Input Tokens: 4133
on_llm_end: LLM Output Tokens: 267
on_llm_end: LLM Output Tokens: 201
on_llm_end: LLM Output Tokens: 171
on_llm_end: LLM Output Tokens: 296
on_llm_end: LLM Output Tokens: 100
on_llm_start: LLM Call #23, Input Tokens: 4128
on_llm_start: LLM Call #24, Input Tokens: 3941on_llm_start: LLM Call #25, Input Tokens: 4131
on_llm_start: LLM Call #26, Input Tokens: 3850

on_llm_end: LLM Output Tokens: 315
on_llm_start: LLM Call #27, Input Tokens: 4127
on_llm_start: L

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,What is the OCR failure rate in European legac...,[MLOps Audit Q4: European division legacy bran...,The OCR failure rate in European legacy branch...,The OCR failure rate is 15%.,1.0,0.948398,1.0
1,Why do OCR failures peak on Tuesdays?,[OCR failures peak on Tuesdays due to weekly b...,OCR failures peak on Tuesdays due to weekly bu...,Due to weekly bulk-batch processing of handwri...,1.0,0.933748,1.0
2,What is the codename for the upgrade project?,[The upgrade project is codenamed 'Project Vis...,Project Vision,The project codename is 'Project Vision'.,1.0,0.730259,1.0


The `judge_llm` is correctly initialized. The `ModuleNotFoundError` is occurring within `ragas` itself, specifically during its internal import of `langchain_community.chat_models.vertexai`. To verify if the module is generally accessible, let's try importing it directly.

In [59]:
results_df.to_csv("Results.csv")

In [60]:
# After all operations are complete, retrieve and display token counts
print("\n--- Token Usage Summary ---")
usage_stats = token_counter.get_token_counts()
for key, value in usage_stats.items():
    print(f"{key.replace('_', ' ').capitalize()}: {value}")



--- Token Usage Summary ---
Llm input tokens: 105155
Llm output tokens: 5429
Llm total tokens: 110584
Llm calls: 34
Embedding tokens: 1670
Embedding calls: 1


In [61]:
try:
    from langchain_community.chat_models import vertexai
    print("Successfully imported langchain_community.chat_models.vertexai")
except ModuleNotFoundError as e:
    print(f"Error importing langchain_community.chat_models.vertexai: {e}")
    print("This indicates a persistent issue with the installation or path of the VertexAI module for LangChain.")

Successfully imported langchain_community.chat_models.vertexai
